# User Delegation To Agents

This notebook demonstrates the CLI/session delegation model: an agent can act on behalf of a user, but operations are constrained by the intersection of the agent's permissions and the delegated user's permissions.

Delegation is currently a CLI/session feature. The HTTP API supports bearer-token agents, but it does not yet expose `delegate` and `undelegate` endpoints.

## 1. Run A Scripted CLI Session

The helper below drives the interactive `mdfs` binary through stdin so the notebook remains reproducible.

In [ ]:
from pathlib import Path
import os
import re
import shutil
import subprocess


def find_repo_root(start=None):
    """Find the mdfs repo even when Jupyter starts in examples/notebooks."""
    env_root = os.environ.get("MDFS_REPO_ROOT") or os.environ.get("MARKDOWNFS_REPO_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())

    start_path = Path(start or Path.cwd()).resolve()
    candidates.extend([start_path, *start_path.parents])

    for candidate in candidates:
        if (candidate / "Cargo.toml").exists() and (candidate / "src").is_dir():
            return candidate

    raise RuntimeError(
        "Could not find the mdfs repository root. "
        "Run the notebook from inside the repo or set MDFS_REPO_ROOT."
    )


def find_cargo():
    """Find Cargo even when the Jupyter kernel has a minimal PATH."""
    candidates = []
    env_cargo = os.environ.get("CARGO")
    if env_cargo:
        candidates.append(Path(env_cargo).expanduser())

    path_cargo = shutil.which("cargo")
    if path_cargo:
        candidates.append(Path(path_cargo))

    candidates.extend([
        Path.home() / ".cargo" / "bin" / "cargo",
        Path("/opt/homebrew/bin/cargo"),
        Path("/usr/local/bin/cargo"),
    ])

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return str(candidate)

    raise RuntimeError(
        "Could not find Cargo. Install Rust from https://rustup.rs, "
        "or set CARGO=/absolute/path/to/cargo before running this notebook."
    )


ROOT = find_repo_root()
CARGO = find_cargo()
DATA_DIR = ROOT / ".demo" / "notebook-delegation"
print(f"Using mdfs repo at {ROOT}")
print(f"Using cargo at {CARGO}")

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)

def run_mdfs_script(lines):
    env = os.environ.copy()
    env["MARKDOWNFS_DATA_DIR"] = str(DATA_DIR)
    env["PATH"] = os.pathsep.join([str(Path(CARGO).parent), env.get("PATH", "")])
    script = "\n".join(lines) + "\n"
    return subprocess.run(
        [CARGO, "run", "--quiet", "--bin", "markdownfs"],
        cwd=ROOT,
        env=env,
        input=script,
        text=True,
        capture_output=True,
        timeout=60,
    )

## 2. Create Users, An Agent, And Delegated Work

The session creates:

- `alice`, the first admin user
- `bob`, a human user whose work can be delegated
- `research-bot`, an agent with a bearer token
- one private file where delegation should fail
- one shared file where delegation should succeed

In [ ]:
result = run_mdfs_script([
    "alice",
    "su root",
    "adduser bob",
    "addagent research-bot",
    "mkdir /delegation",
    "chmod 777 /delegation",
    "touch /delegation/private-note.md",
    "chown bob:bob /delegation/private-note.md",
    "chmod 600 /delegation/private-note.md",
    "touch /delegation/shared-note.md",
    "chown bob:bob /delegation/shared-note.md",
    "chmod 666 /delegation/shared-note.md",
    "su research-bot",
    "delegate bob",
    "write /delegation/private-note.md # This should fail because the agent has no write bit",
    "write /delegation/shared-note.md # Delegated Research Note",
    "stat /delegation/shared-note.md",
    "cat /delegation/shared-note.md",
    "commit delegated research note",
    "log",
    "exit",
])

print("STDOUT:\n")
print(result.stdout)
print("STDERR:\n")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("scripted mdfs session failed")

## 3. Pull Out The Agent Token

The raw token is printed once. In a real workflow, save it immediately and pass it as `Authorization: Bearer <token>` to the HTTP API.

In [ ]:
token_match = re.search(r"API token \(save this .*?\):\n\s+([a-f0-9]+)", result.stdout)
agent_token = token_match.group(1) if token_match else None
print(agent_token)

## 4. What To Notice

- `delegate bob` changes the effective ownership of successful writes to Bob.
- The write to `private-note.md` fails because Bob can write it, but the agent cannot; delegation uses least privilege.
- The write to `shared-note.md` succeeds because both Bob and the agent have write permission.
- The commit gives the handoff a recoverable version boundary.